# Exercise 3: Install a skill and use it on the app

**Goal:** After this exercise, you can install a skill into a project and have Codex use it for one change to the app.

**Scenario:** The planning-with-files skill makes an agent keep its plan and progress in files. You install it, then ask Codex to plan and make one change with it.

**Coding agent:** yes. You use Codex in a terminal.

**Where you work:** your own copy of the app, the folder `week-02/day-01/exercise-03-use-a-skill/reference-app` in the course folder. You run the cells of this notebook and four commands in a terminal. You do not commit: the reset puts the folder back.

**Done when:** cell 4 prints `Done:`.

**Optional part:** none.

**What you need to know first:**

- A skill is a folder of instructions, with a `SKILL.md` file, that an agent loads only when a task needs it.
- In Codex, `$` and the skill's name in a request tell Codex to use that skill.

**What you have:** this exercise's `reference-app` folder is your copy of the app, with exercises 1 and 2's work. Its `AGENTS.md` tells Codex to run the tests.

**Start from:** step 1 and cell 1. To start again, open a terminal in the course folder. Run `source .venv/bin/activate`, then `python reset.py exercise-03-use-a-skill`. The reset puts this folder back as the course supplied it.

### Step 1. See the start

**Before running cell 1:** open **Terminal > New Terminal** in the course folder (the folder containing `pyproject.toml`). Run:

```sh
uv sync
```

This installs the packages for both weeks, including Flask, into the course folder's `.venv`. The first run needs internet access. All three week 2 exercises use this same environment.

**Select the kernel:** click **Select Kernel** at the top right, then **Python Environments**, and select the course folder's `.venv/bin/python`.

**Cannot see it?** Refresh the environment picker after `uv sync`. If it is still missing, run **Python: Select Interpreter** from the Command Palette, choose **Enter interpreter path**, and browse to `.venv/bin/python` in the course folder. Then reopen the notebook's kernel picker and select it. If the notebook already ran with another kernel, restart it and run from cell 1.

Run cell 1. It shows the skills in your copy and the skill's plan file.

You should see `Skills in .agents/skills: none` and `Plan files: none`.

In [2]:
# 1. See the start. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
from pathlib import Path

print("Skills in .agents/skills:", ", ".join(p.name for p in Path(".agents/skills").glob("*")) or "none")
print("Plan files:", ", ".join(p.name for p in Path(".").glob("*.md") if p.stem in ("task_plan", "findings", "progress")) or "none")

Skills in .agents/skills: none
Plan files: none


### Step 2. Read the skill before you install it

Open the skill's `SKILL.md` on GitHub: https://github.com/OthmanAdi/planning-with-files/blob/master/skills/planning-with-files/SKILL.md. GitHub shows the file as a page. Its `name` and its `description` are at the top, before the first heading. Read the `name` and the first two sentences of the `description`.

| Read | What it tells you | Why it matters |
|---|---|---|
| `name: planning-with-files` | The name you type after `$` in step 4 | Codex finds a skill by its name. |
| "Keeps task_plan.md, findings.md, and progress.md on disk" | The skill's output: three plain files in your folder | You can read and check each file. "We recommend installing skills only from trusted sources." Read what a skill does before you install it. |

### Step 3. Install the skill into your copy

Open a terminal: **Terminal > New Terminal**. It opens in the course folder. Run these three commands, one at a time:

```sh
source .venv/bin/activate
cd week-02/day-01/exercise-03-use-a-skill/reference-app
npx --yes skills@1.7.4 add OthmanAdi/planning-with-files --skill planning-with-files --agent codex --yes
```

1. The first command starts the course's shared Python environment in this terminal. Codex then runs `pytest` with the app's Python.
2. The second command goes into the folder you work in.
3. The third command downloads the skill from GitHub into `.agents/skills/planning-with-files` in this folder. `npx` comes with Node.js: it downloads a Node package, here the install tool `skills`, and runs it without installing it globally. `npx --yes` runs the install tool, version 1.7.4, with no question. `--agent codex` puts the skill where Codex reads skills. The last `--yes` answers the tool's own questions.

The command has no `-g`, so the skill goes into this folder only. "For repositories, Codex scans .agents/skills in every directory from your current working directory up to the repository root."

You should see `Installed 1 skill` and `Done!`. If the terminal says `npx: command not found`, tell your trainer: the workspace needs Node.js. Then run cell 2. You should see `name: planning-with-files`.

In [3]:
# 2. See the installed skill. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
from pathlib import Path

skill = Path(".agents/skills/planning-with-files")
if not (skill / "SKILL.md").is_file():
    print("The skill is not in .agents/skills yet. Do step 3.")
else:
    print((skill / "SKILL.md").read_text(encoding="utf-8").splitlines()[1])
    print("Scripts it brings:", len(list((skill / "scripts").glob("*"))))

name: planning-with-files
Scripts it brings: 22


### Step 4. Use the skill for one change

In the same terminal, start a new Codex session:

```sh
codex
```

If Codex asks if you trust this folder, answer yes. The course gave you this folder.

Write your request to Codex in your own words, and press Enter. It must say these things:

| What your request says | Why |
|---|---|
| It starts with `$planning-with-files` | `$` and the name tell Codex to use the skill: "In Codex CLI or the IDE extension, run /skills or type $ to mention a skill." |
| Plan the change, then make it. The change: on the case list page, the words Review first next to each case with a selection score of 50 or more | The skill keeps the plan in its three files. The check looks for these words and this score. |

<details><summary>Stuck? One way to say it</summary>

`$planning-with-files Plan and make this change: on the case list page, show the words Review first next to each case whose selection score is 50 or more.`

</details>

Codex writes the skill's three files, makes the change and runs `pytest`. If it asks to start the app or to use the network, answer no. When Codex reports its change, type `/quit` and press Enter. This stops Codex.

### Step 5. See the skill's output

Run cell 3. It lists the skill's three files and the plan's phases.

You should see three file names with their line counts, and phases for the Review first change.

In [4]:
# 3. See the skill's output. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
from pathlib import Path

for p in [*Path(".").glob("*.md"), *Path(".planning").glob("*/*.md")]:
    if p.stem in ("task_plan", "findings", "progress"):
        lines = p.read_text(encoding="utf-8").splitlines()
        print(f"{p}: {len(lines)} lines")
        print("\n".join([line for line in lines if "Phase" in line][:6]) if p.stem == "task_plan" else "", end="")

progress.md: 12 lines
findings.md: 10 lines
task_plan.md: 47 lines
## Current Phase
## Phases
### Phase 1: Requirements & Discovery
### Phase 2: Planning & Structure
### Phase 3: Implementation
### Phase 4: Testing & Verification

### Step 6. Check

Run cell 4. It prints `Done:` when four things are true:

- The skill is in `.agents/skills/planning-with-files`.
- The skill's three files are in your copy, written after the install.
- The case list shows Review first next to each case with a score of 50 or more, and no other.
- The app's tests pass.

Otherwise it prints `Not yet:` and each thing that is still missing. You do not need to read its code.

In [5]:
# 4. Check. You do not need to read this code.
%cd -q {_dh[0]}/reference-app
import subprocess
import sys
from datetime import date
from pathlib import Path


def plan_file(name: str) -> Path | None:
    """Return where the skill wrote this file, or None."""
    found = [p for p in [Path(name), *Path(".planning").glob(f"*/{name}")] if p.is_file()]
    return found[0] if found else None


def check() -> list[str]:
    """Return what is left: one line for each thing that is not yet right."""
    left = []
    skill = Path(".agents/skills/planning-with-files/SKILL.md")
    if not skill.is_file():
        left.append("The skill is not in .agents/skills/planning-with-files. Do step 3.")
    installed = skill.stat().st_mtime if skill.is_file() else 0.0
    missing = [name for name in ("task_plan.md", "findings.md", "progress.md")
               if not (plan_file(name) and plan_file(name).stat().st_mtime > installed)]
    if missing:
        left.append(f"The skill's files are not written yet: {', '.join(missing)}. Do step 4.")
    for name in ("app", "cases"):
        sys.modules.pop(name, None)  # read the files as they are now
    try:
        import app as workbench
        import cases
    except ImportError:
        return ["Flask is missing. Run uv sync in the course folder, select its .venv kernel, and restart from cell 1."]
    rows = workbench.app.test_client().get("/").get_data(as_text=True).split("<tr>")[2:]
    wrong = [v["case_id"] for v in cases.case_list(cases.load_cases(), date.today())
             if ("Review first" in next((r for r in rows if v["case_id"] in r), "")) != (v["score"] >= 50)]
    if wrong:
        left.append(f"Rows of the case list with the label wrong: {len(wrong)} ({wrong[0]}). Do step 4.")
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                           capture_output=True, text=True)
    if tests.returncode:
        last = (tests.stdout.strip() or tests.stderr.strip()).splitlines()[-1]
        left.append(f"The app's tests do not pass ({last}). Ask Codex to fix what it changed.")
    return left


left = check()
print(f"Not yet: {len(left)} left." if left else "Done: the skill is installed in your copy, its "
      "three files are written, the case list shows Review first, and the tests pass.")
for line in left:
    print("- " + line)

Done: the skill is installed in your copy, its three files are written, the case list shows Review first, and the tests pass.


### Afterwards, together

1. "A new session with nothing loaded knows nothing about your project." Why does a plan in a file help a new session more than the old session's history?
2. "Skills for every repository you work in go in your home folder's `.agents/skills`." When would you install a skill there, for every project, and why?
3. The skill brings scripts that run with Codex's permissions. What would you check before you install a skill on a project at work?

<details><summary>Answer</summary>

The install: with no `-g`, the command puts the skill into this folder's `.agents/skills`, where Codex finds it for this app only. Nothing outside the folder changes. The version number keeps the command the same for every learner.

Read the skill first: a skill runs with the agent's permissions, so read what it does before you install it.

The request: `$` and the name tell Codex to use the skill. The skill then writes `task_plan.md`, `findings.md` and `progress.md`, and Codex makes the change and runs the tests.

The finished folder, with the skill's three files, is in [`answers/exercise-03-use-a-skill/reference-app`](../answers/exercise-03-use-a-skill/reference-app/). Each step and its reason is also in [`answers/exercise-03-use-a-skill/decisions.md`](../answers/exercise-03-use-a-skill/decisions.md).

</details>